In [1]:
# ============================================================
#  SEQUENTIAL FEATURE EXTRACTION — WITH TIMING
#  Platform : Kaggle  |  GPU : T4 x2
#  Mode     : Models run ONE AFTER ANOTHER (sequential)
# ============================================================

import numpy as np
import os, time, warnings
import tensorflow as tf
warnings.filterwarnings('ignore')

from tensorflow.keras.applications import MobileNetV2, VGG16, DenseNet121
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as pre_mobile
from tensorflow.keras.applications.vgg16        import preprocess_input as pre_vgg
from tensorflow.keras.applications.densenet     import preprocess_input as pre_densenet
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import Model

# ── Kaggle dataset path (update if needed) ──────────────────
TRAIN_DIR = '/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Training'
TEST_DIR  = '/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Testing'
IMG_SIZE  = (224, 224)

# ── GPU check ───────────────────────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs detected: {len(gpus)}")
for g in gpus:
    print(" ", g)

# ── Build feature extractor from any base model ─────────────
def build_extractor(base_model_fn, weights='imagenet'):
    base = base_model_fn(
        weights=weights,
        include_top=False,
        input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
    )
    base.trainable = False
    out = tf.keras.layers.GlobalAveragePooling2D()(base.output)
    return Model(inputs=base.input, outputs=out)

# ── Extract features from a directory ───────────────────────
def extract_features(model, preprocess_fn, data_dir):
    features, labels = [], []
    for cls in sorted(os.listdir(data_dir)):
        cls_path = os.path.join(data_dir, cls)
        if not os.path.isdir(cls_path) or cls.startswith('.'):
            continue
        files = [f for f in os.listdir(cls_path)
                 if f.lower().endswith(('.jpg','.jpeg','.png','.bmp'))]
        for fname in files:
            try:
                img = image.load_img(os.path.join(cls_path, fname),
                                     target_size=IMG_SIZE)
                arr = image.img_to_array(img)
                arr = np.expand_dims(arr, axis=0)
                arr = preprocess_fn(arr)
                feat = model.predict(arr, verbose=0).flatten()
                features.append(feat)
                labels.append(cls)
            except Exception as e:
                print(f"  Skipping {fname}: {e}")
    return np.array(features), np.array(labels)

# ════════════════════════════════════════════════════════════
#  SEQUENTIAL EXECUTION
#  Each model loads, extracts train+test, then the next starts
# ════════════════════════════════════════════════════════════

timing_log = {}   # stores per-model times

print("\n" + "="*60)
print("  SEQUENTIAL FEATURE EXTRACTION")
print("="*60)

overall_start = time.perf_counter()

# ── Model 1: MobileNetV2 ─────────────────────────────────────
print("\n[1/3] MobileNetV2 — loading...")
t0 = time.perf_counter()

mobilenet = build_extractor(MobileNetV2)
print("  Extracting train features...")
f_mobile_train, labels_train = extract_features(mobilenet, pre_mobile, TRAIN_DIR)
print("  Extracting test features...")
f_mobile_test,  labels_test  = extract_features(mobilenet, pre_mobile, TEST_DIR)
del mobilenet       # free GPU memory before next model
tf.keras.backend.clear_session()

t_mobile = time.perf_counter() - t0
timing_log['MobileNetV2'] = t_mobile
print(f"  ✅ MobileNetV2 done  | time: {t_mobile:.2f}s"
      f"  | train shape: {f_mobile_train.shape}")

# ── Model 2: VGG16 ───────────────────────────────────────────
print("\n[2/3] VGG16 — loading...")
t0 = time.perf_counter()

vgg = build_extractor(VGG16)
print("  Extracting train features...")
f_vgg_train, _ = extract_features(vgg, pre_vgg, TRAIN_DIR)
print("  Extracting test features...")
f_vgg_test,  _ = extract_features(vgg, pre_vgg, TEST_DIR)
del vgg
tf.keras.backend.clear_session()

t_vgg = time.perf_counter() - t0
timing_log['VGG16'] = t_vgg
print(f"  ✅ VGG16 done        | time: {t_vgg:.2f}s"
      f"  | train shape: {f_vgg_train.shape}")

# ── Model 3: DenseNet121 ─────────────────────────────────────
print("\n[3/3] DenseNet121 — loading...")
t0 = time.perf_counter()

densenet = build_extractor(DenseNet121)
print("  Extracting train features...")
f_dense_train, _ = extract_features(densenet, pre_densenet, TRAIN_DIR)
print("  Extracting test features...")
f_dense_test,  _ = extract_features(densenet, pre_densenet, TEST_DIR)
del densenet
tf.keras.backend.clear_session()

t_dense = time.perf_counter() - t0
timing_log['DenseNet121'] = t_dense
print(f"  ✅ DenseNet121 done  | time: {t_dense:.2f}s"
      f"  | train shape: {f_dense_train.shape}")

# ── Total sequential time ────────────────────────────────────
sequential_total = time.perf_counter() - overall_start
timing_log['Sequential_Total'] = sequential_total

# ── Save arrays for comparison later ────────────────────────
np.save('/kaggle/working/seq_mobile_train.npy', f_mobile_train)
np.save('/kaggle/working/seq_vgg_train.npy',    f_vgg_train)
np.save('/kaggle/working/seq_dense_train.npy',  f_dense_train)
np.save('/kaggle/working/seq_mobile_test.npy',  f_mobile_test)
np.save('/kaggle/working/seq_vgg_test.npy',     f_vgg_test)
np.save('/kaggle/working/seq_dense_test.npy',   f_dense_test)
np.save('/kaggle/working/labels_train.npy',     labels_train)
np.save('/kaggle/working/labels_test.npy',      labels_test)

# ════════════════════════════════════════════════════════════
#  TIMING REPORT
# ════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("  SEQUENTIAL TIMING REPORT")
print("="*60)
print(f"  MobileNetV2  : {timing_log['MobileNetV2']:.2f} seconds")
print(f"  VGG16        : {timing_log['VGG16']:.2f} seconds")
print(f"  DenseNet121  : {timing_log['DenseNet121']:.2f} seconds")
print(f"  {'─'*38}")
print(f"  TOTAL        : {sequential_total:.2f} seconds")
print("="*60)

# Save timing to disk so parallel notebook can compare
import json
with open('/kaggle/working/sequential_timing.json', 'w') as f:
    json.dump(timing_log, f, indent=2)
print("\n✅ Timing saved to /kaggle/working/sequential_timing.json")

2026-04-18 18:12:51.383755: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776535971.777867      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776535971.880371      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776535972.808464      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776535972.808503      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776535972.808505      55 computation_placer.cc:177] computation placer alr

GPUs detected: 2
  PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')
  PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')

  SEQUENTIAL FEATURE EXTRACTION

[1/3] MobileNetV2 — loading...


I0000 00:00:1776536009.393117      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1776536009.398967      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
  Extracting train features...


I0000 00:00:1776536013.647452     128 service.cc:152] XLA service 0x79cbb8002df0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776536013.647488     128 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1776536013.647491     128 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1776536014.313937     128 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-04-18 18:13:43.284773: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-18 18:13:43.419287: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
I0000 00:00:1776536024.781382     128 device_co

  Extracting test features...
  ✅ MobileNetV2 done  | time: 655.43s  | train shape: (5600, 1280)

[2/3] VGG16 — loading...
58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
  Extracting train features...
  Extracting test features...
  ✅ VGG16 done        | time: 674.02s  | train shape: (5600, 512)

[3/3] DenseNet121 — loading...
29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
  Extracting train features...
  Extracting test features...
  ✅ DenseNet121 done  | time: 696.38s  | train shape: (5600, 1024)

  SEQUENTIAL TIMING REPORT
  MobileNetV2  : 655.43 seconds
  VGG16        : 674.02 seconds
  DenseNet121  : 696.38 seconds
  ──────────────────────────────────────
  TOTAL        : 2025.84 seconds

✅ Timing saved to /kaggle/working/sequential_timing.json
